In [ ]:
import os
import sys
import dotenv
import pandas as pd
import torch
import gc
import matplotlib.pyplot as plt
from diffusers import AutoPipelineForText2Image

sys.path.append('..')
dotenv.load_dotenv()
os.environ['WANDB_DISABLED'] = "true"
assert len(os.getenv('HF_TOKEN'))>0

from vision_unlearning.datasets import download_dataset_lfw
from vision_unlearning.utils.logger import get_logger, setup_loggers
from vision_unlearning.utils.gradient_weighting import GradientWeightingMethodSimple


sys.path.append('../../TRDP-unlearning')  # Sorry, FADE is still in a closed-source repo...
from unlearner_lora_distillation import UnlearnerLoraDistillation

In [ ]:
num_train_epochs = 5
model_base_name = "CompVis/stable-diffusion-v1-4"
model_lora_path = f"assets/models/fade_Bush_to_Blair_{num_train_epochs:03d}"
hub_model_id = "LeonardoBenitez/demo-vision-unlearning-fade"
dataset_base_path = 'assets/datasets/lfw_splits'

target = "George_W_Bush"
target_overwrite = "Tony_Blair"

validation_prompt = f'An image of {target}'
example_prompts_forget = [
    f'An image of {target}',
    f'Photograph of {target.replace("_", " ")}; high definition',
]
example_prompts_retain = [
    f'An image of {target_overwrite}',
    f'Photograph of {target_overwrite.replace("_", " ")}; high definition',
]

### No need to change anything from now on... ###
logger = get_logger('main')
setup_loggers(modules_info=['vision_unlearning.'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'

dataset_forget_name = f"{dataset_base_path}/{target}/train_forget"
dataset_retain_name = f"{dataset_base_path}/{target}/train_retain"

logger.info(f"Overwriting {target} by the class {target_overwrite}")

# Original model
As you can see, it can generate the undesired concept

In [ ]:
pipeline = AutoPipelineForText2Image.from_pretrained(model_base_name, torch_dtype=torch.float16, safety_checker=None).to(device)

In [ ]:
for prompt in example_prompts_forget + example_prompts_retain:
    image = pipeline(prompt).images[0]
    plt.imshow(image)
    plt.title(prompt)
    plt.show()

In [ ]:
del pipeline
gc.collect()
torch.cuda.empty_cache()

# Prepare dataset for unlearning
images of the concept to be forgotten + some random images of concepts to be retained

In [ ]:
# !rm -r {dataset_base_path}
if not os.path.exists(dataset_base_path):
    download_dataset_lfw(dataset_forget_name, dataset_retain_name, target)
!find {dataset_forget_name} -type f | wc -l

# Unlearning
Performing the actual model update

In [ ]:
!nvidia-smi

In [ ]:
free_memory, total_memory = torch.cuda.mem_get_info()  # in GB
logger.info(f"Free Memory: {free_memory / 1e9:.2f} GB")
logger.info(f"Total Memory: {total_memory / 1e9:.2f} GB")

# Most hyperparameters should be set here, excep the ones that change at runtime
hyperparameters = {
    "dataloader_num_workers": 2,
    "resolution": 512,
    "num_validation_images": 1,

    "mixed_precision": "no",
    "learning_rate": 1e-4,
    "max_grad_norm": 1.0,
    "lr_scheduler_type": "lr_scheduler",
    "lr_warmup_steps": 0,
    "num_train_epochs": num_train_epochs,
    "validation_epochs": 1,
    "checkpointing_steps": 10000,
    "lr_scheduler_type": "cosine",
    "logging_steps": 20,
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "random_flip": True,

    "lora_r": 4,
    "target_modules": ["to_k", "to_q", "to_v", "to_out.0"],
    "lora_alpha": 4,
    "lora_dropout": 0.1,

    "seed": 42,
}

if free_memory > 20e9:
    hyperparameters.update({
        "per_device_train_batch_size": 4,
        "gradient_accumulation_steps": 1,
    })
elif free_memory > 14e9:
    hyperparameters.update({
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 2,
    })
else:
    logger.error('Too little GPU, diverting power from life support...')
    hyperparameters.update({
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 4,
    })

logger.info(hyperparameters)

In [ ]:
unlearner = UnlearnerLoraDistillation(
    model_name_or_path=model_base_name,
    dataset_forget_name=dataset_forget_name,
    dataset_retain_name=dataset_retain_name,
    output_dir=model_lora_path,
    overwritting_concept = target_overwrite,
    validation_prompt=f"An image of {target}",
    final_eval_prompts_forget = example_prompts_forget,
    final_eval_prompts_retain = example_prompts_retain,
    gradient_weighting_method = GradientWeightingMethodSimple(forget_weight=0.3, retain_weight=1.0),
    hub_model_id = hub_model_id,
    **hyperparameters,
)

eval_results = unlearner.train()
pd.DataFrame([{'Name': r.metric_name, 'Value': r.metric_value} for r in eval_results])

In [ ]:
del unlearner
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Show validation prompts
for epoch in range(0, num_train_epochs, 1):
    plt.imshow(plt.imread(os.path.join(model_lora_path, 'images', f'val_prompt_{epoch:02}_01.png')))
    plt.title(f'Epoch {epoch}')
    plt.show()

# Check unlearned model
as you can see, it does NOT generate the undesired concept anymore

In [ ]:
pipeline = AutoPipelineForText2Image.from_pretrained(model_base_name, torch_dtype=torch.float16, safety_checker=None).to(device)
pipeline.load_lora_weights(model_lora_path, weight_name='pytorch_lora_weights.safetensors')

In [ ]:
for prompt in example_prompts_forget + example_prompts_retain:
    image = pipeline(prompt).images[0]
    plt.imshow(image)
    plt.title(prompt)
    plt.show()